# Assignment 1, Part A: Triton Kernel Warm-Up

**Purpose:** get familiar with the course instance and learn to write, validate, and benchmark custom GPU kernels in Triton before optimizing real training workloads in Part B.

You will implement four kernels:

| # | Kernel | Direction |
|---|--------|-----------|
| (a) | Elementwise addition $c = a + b$ | forward (backward given) |
| (b) | RMSNorm | forward and backward |
| (c) | SwiGLU activation | forward and backward |
| (d) | Cross-entropy loss (fused log-softmax + NLL) | forward and backward |

**How this notebook works**

- Kernel (a) ships as a worked skeleton. The launch logic and the `torch.autograd.Function` wrapper are provided, and the kernel body is a `TODO` stub that writes zeros, so its tests run and fail before you write anything. Use it as the template for the rest.
- For kernels (b), (c), (d) you write everything, both the Triton kernels and the launch logic / `torch.autograd.Function` wrappers. The signatures, eager references, and tests are given. The bodies are `...` placeholders, so those cells and their tests will error until you implement them.
- Your job, per kernel:
  1. **Validate** against the PyTorch eager reference (max abs/rel error in fp32 and bf16).
  2. **Generalize** with masking so arbitrary non-power-of-two shapes work, up to real vocab sizes for cross-entropy.
  3. **Benchmark** against the eager baseline. Report achieved bandwidth (GB/s) and speedup, and state whether the op is memory- or compute-bound.
  4. **Tune** with `triton.autotune` or a manual sweep over `BLOCK` / `num_warps`.
- Run cells in order. Kernel (a)'s tests are supposed to fail until you implement it, and the (b), (c), (d) cells will error until you do. Your final submitted notebook should have every test reporting `PASS` and a filled-in results table at the bottom.

Run this on the course H100 instance (`torch.cuda.is_available()` must be true).


In [ ]:
import torch
import torch.nn.functional as F
import triton
import triton.language as tl

assert torch.cuda.is_available(), "This notebook needs a CUDA GPU (use the course H100 instance)."
torch.manual_seed(0)
DEV = "cuda"
print("torch", torch.__version__, "| triton", triton.__version__, "| device:", torch.cuda.get_device_name(0))


## Provided helpers

`report(...)` compares your kernel's output with the reference (`torch.allclose` semantics + printed max abs/rel error) and records pass/fail. `bench_op(...)` times your kernel and the eager baseline with `triton.testing.do_bench` and reports achieved bandwidth for a minimum-traffic model (`nbytes` = bytes that must move if the kernel were perfectly fused). You just call them. No edits needed.


In [ ]:
ALL_OK = {}
RESULTS = []

def report(name, out, ref, atol, rtol):
    """Compare out vs ref, then print and record pass/fail."""
    o, r = out.float(), ref.float()
    abs_err = (o - r).abs().max().item()
    rel_err = ((o - r).abs() / r.abs().clamp_min(1e-6)).max().item()
    ok = torch.allclose(o, r, atol=atol, rtol=rtol)
    ALL_OK[name] = ok
    status = "PASS" if ok else "FAIL (kernel not implemented yet?)"
    print(f"{name:<34} max abs err = {abs_err:.3e}   max rel err = {rel_err:.3e}   [{status}]")

def bench_op(op, shape_str, ours_fn, ref_fn, nbytes):
    """Time ours_fn vs ref_fn. nbytes is the minimum-traffic byte count."""
    t_ours = triton.testing.do_bench(ours_fn, warmup=25, rep=100)
    t_ref = triton.testing.do_bench(ref_fn, warmup=25, rep=100)
    g_ours = nbytes / (t_ours * 1e-3) / 1e9
    g_ref = nbytes / (t_ref * 1e-3) / 1e9
    RESULTS.append((op, shape_str, t_ours, t_ref, g_ours, g_ref, t_ref / t_ours))
    print(f"{op:<26} {shape_str:<22} ours {t_ours:8.3f} ms ({g_ours:7.1f} GB/s) | "
          f"eager {t_ref:8.3f} ms ({g_ref:7.1f} GB/s) | speedup {t_ref / t_ours:5.2f}x")

def results_table():
    print(f"{'op':<26} {'shape':<22} {'ours (ms)':>10} {'eager (ms)':>10} "
          f"{'GB/s ours':>10} {'GB/s eager':>11} {'speedup':>8}")
    for op, shp, t_o, t_r, g_o, g_r, sp in RESULTS:
        print(f"{op:<26} {shp:<22} {t_o:>10.3f} {t_r:>10.3f} {g_o:>10.1f} {g_r:>11.1f} {sp:>7.2f}x")

def peak_mem_mib(fn):
    """Peak CUDA memory (MiB) while running fn()."""
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    fn()
    return torch.cuda.max_memory_allocated() / 2**20


## Kernel (a): elementwise addition

$$c = a + b, \qquad a, b, c \in \mathbb{R}^N$$

This is the canonical memory-bound op. It moves $2N$ elements in and $N$ out while doing only one FLOP per element (arithmetic intensity $\approx 1/12$ FLOP/byte in fp32). It is also the warm-up for the pattern everything below uses. Grid over blocks, masked load, compute, masked store.

Fill in `add_kernel`. The `Add` autograd wrapper is provided. Note how `backward` just passes the gradient through, since $\partial c/\partial a = \partial c/\partial b = 1$. The wrapper is also the exact pattern you will need for RMSNorm's and cross-entropy's custom backward.


In [ ]:
@triton.jit
def add_kernel(a_ptr, b_ptr, out_ptr, n_elements, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n_elements
    # TODO (student)
    stub = tl.zeros((BLOCK,), dtype=tl.float32)
    tl.store(out_ptr + offs, stub.to(out_ptr.dtype.element_ty), mask=mask)


class Add(torch.autograd.Function):
    """Differentiable wrapper around add_kernel (c = Add.apply(a, b))."""

    @staticmethod
    def forward(ctx, a, b):
        a, b = a.contiguous(), b.contiguous()
        out = torch.empty_like(a)
        n = out.numel()
        grid = (triton.cdiv(n, 1024),)
        add_kernel[grid](a, b, out, n, BLOCK=1024)
        return out

    @staticmethod
    def backward(ctx, grad_out):
        # y = a + b has dy/da = dy/db = 1, so just pass the gradient on.
        return grad_out.contiguous(), grad_out.contiguous()


def triton_add(a, b):
    return Add.apply(a, b)


**Correctness.** Odd sizes exercise your masking. The sliced tensor exercises the non-contiguous path. The wrapper calls `.contiguous()`, so think about what that costs and how a strided kernel could avoid it.


In [ ]:
for dtype, atol, rtol in [(torch.float32, 1e-6, 1e-5), (torch.bfloat16, 1e-2, 1e-2)]:
    for n in [1, 1023, 1_048_575, 10_000_003]:
        a = torch.randn(n, device=DEV, dtype=dtype)
        b = torch.randn(n, device=DEV, dtype=dtype)
        report(f"add n={n} {str(dtype).split('.')[-1]}", triton_add(a, b), a + b, atol, rtol)

a = torch.randn(1024, 1025, device=DEV)[:, 1:]   # non-contiguous view
b = torch.randn_like(a)
report("add non-contiguous", triton_add(a, b), a + b, 1e-6, 1e-5)

# autograd plumbing: sum((a+b)^2) has d/da = d/db = 2(a+b)
a = torch.randn(1000, device=DEV, requires_grad=True)
b = torch.randn(1000, device=DEV, requires_grad=True)
triton_add(a, b).pow(2).sum().backward()
g_ref = 2 * (a.detach() + b.detach())
report("add autograd a.grad", a.grad, g_ref, 1e-6, 1e-5)
report("add autograd b.grad", b.grad, g_ref, 1e-6, 1e-5)


**Benchmark.** ~3 GiB of traffic per call in fp32. What fraction of the H100's ~3.35 TB/s HBM3 bandwidth do you and eager PyTorch each hit?


In [ ]:
N = 256 * 1024 * 1024
a = torch.randn(N, device=DEV)
b = torch.randn(N, device=DEV)
nbytes = 3 * N * a.element_size()          # read a, read b, write out
bench_op("add", f"N={N}", lambda: triton_add(a, b), lambda: a + b, nbytes)
del a, b
torch.cuda.empty_cache()


## Kernel (b): RMSNorm (forward and backward)

**TODO (student):** derive the forward and backward math for RMSNorm in this cell before implementing. The handout gives the definition. The eager reference `torch_rmsnorm` below fixes the exact semantics (fp32 math, $y = x \cdot \mathrm{rsqrt}(\mathrm{mean}(x^2) + \epsilon) \cdot w$, cast back to the input dtype).


In [ ]:
@triton.jit
def rmsnorm_fwd_kernel(x_ptr, w_ptr, out_ptr, rstd_ptr,
                       n_cols, eps,
                       x_stride, out_stride,
                       BLOCK: tl.constexpr):
    # TODO (student)
    ...


@triton.jit
def rmsnorm_bwd_kernel(x_ptr, w_ptr, dy_ptr, rstd_ptr, dx_ptr, dw_ptr,
                       n_cols,
                       x_stride, dy_stride, dx_stride,
                       BLOCK: tl.constexpr):
    # TODO (student)
    ...


class RMSNorm(torch.autograd.Function):
    """Differentiable wrapper around rmsnorm_{fwd,bwd}_kernel."""

    @staticmethod
    def forward(ctx, x, weight, eps=1e-6):
        # TODO (student)
        ...

    @staticmethod
    def backward(ctx, grad_out):
        # TODO (student)
        ...


def triton_rmsnorm(x, weight, eps=1e-6):
    return RMSNorm.apply(x, weight, eps)


def torch_rmsnorm(x, weight, eps=1e-6):
    dtype = x.dtype
    xf = x.float()
    y = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + eps) * weight.float()
    return y.to(dtype)


In [ ]:
for dtype, atol, rtol in [(torch.float32, 1e-5, 1e-4), (torch.bfloat16, 1e-2, 1e-2)]:
    for shape in [(8, 512, 1024), (4096, 1000), (256, 8192)]:   # 1000 is an odd D, 8192 is bigger than one BLOCK
        x = torch.randn(*shape, device=DEV, dtype=dtype)
        w = torch.randn(shape[-1], device=DEV, dtype=dtype)
        report(f"rmsnorm fwd {shape} {str(dtype).split('.')[-1]}",
               triton_rmsnorm(x, w), torch_rmsnorm(x, w), atol, rtol)

for dtype, atol, rtol in [(torch.float32, 1e-4, 1e-3), (torch.bfloat16, 2e-2, 2e-2)]:
    for shape in [(64, 1024), (32, 8192)]:   # 8192 exercises the bigger-than-BLOCK loop in backward too
        x1 = torch.randn(*shape, device=DEV, dtype=dtype, requires_grad=True)
        w1 = torch.randn(shape[-1], device=DEV, dtype=dtype, requires_grad=True)
        x2 = x1.detach().clone().requires_grad_(True)
        w2 = w1.detach().clone().requires_grad_(True)
        torch_rmsnorm(x1, w1).pow(2).sum().backward()
        triton_rmsnorm(x2, w2).pow(2).sum().backward()
        report(f"rmsnorm bwd dx {shape} {str(dtype).split('.')[-1]}", x2.grad, x1.grad, atol, rtol)
        report(f"rmsnorm bwd dw {shape} {str(dtype).split('.')[-1]}", w2.grad, w1.grad, atol, rtol)


In [ ]:
x = torch.randn(8, 4096, 4096, device=DEV, dtype=torch.bfloat16)   # LLM-ish activations
w = torch.randn(4096, device=DEV, dtype=torch.bfloat16)

nbytes_fwd = (2 * x.numel() + w.numel()) * x.element_size()   # read x + w, write y
bench_op("rmsnorm fwd", str(tuple(x.shape)), lambda: triton_rmsnorm(x, w),
         lambda: torch_rmsnorm(x, w), nbytes_fwd)

xg = x.clone().requires_grad_(True)
wg = w.clone().requires_grad_(True)

def triton_rmsnorm_fwd_bwd():
    triton_rmsnorm(xg, wg).sum().backward()
    xg.grad = wg.grad = None

def torch_rmsnorm_fwd_bwd():
    torch_rmsnorm(xg, wg).sum().backward()
    xg.grad = wg.grad = None

nbytes_fb = 2 * nbytes_fwd   # roughly double the traffic for the extra backward pass
bench_op("rmsnorm fwd+bwd", str(tuple(x.shape)), triton_rmsnorm_fwd_bwd, torch_rmsnorm_fwd_bwd, nbytes_fb)
print(f"peak memory:  eager {peak_mem_mib(torch_rmsnorm_fwd_bwd):8.0f} MiB   fused {peak_mem_mib(triton_rmsnorm_fwd_bwd):8.0f} MiB")
del x, w, xg, wg
torch.cuda.empty_cache()


## Kernel (c): SwiGLU (forward and backward)

$$y = \mathrm{SiLU}(a) \odot b, \qquad \mathrm{SiLU}(a) = a\,\sigma(a) = \frac{a}{1 + e^{-a}}$$

In an LLM feed-forward block, $a = xW_1$ and $b = xW_3$ come from cuBLAS GEMMs. Leave those in `torch.matmul` since you will not beat cuBLAS. The $\mathrm{SiLU}(\cdot) \odot (\cdot)$ part is elementwise and memory-bound, and eager PyTorch materializes an intermediate for the SiLU output in both directions. Fuse forward into one elementwise kernel and backward into another.

Given the upstream gradient $g = \partial\mathcal L/\partial y$:

$$\frac{\partial y}{\partial a} = b\,\sigma(a)\big(1 + a(1 - \sigma(a))\big), \qquad \frac{\partial y}{\partial b} = \mathrm{SiLU}(a)$$

so $da = g \cdot \partial y/\partial a$ and $db = g \cdot \mathrm{SiLU}(a)$.

Do the math in fp32 (loads cast up, stores cast down). `tl.sigmoid` is available.


In [ ]:
@triton.jit
def swiglu_fwd_kernel(a_ptr, b_ptr, out_ptr, n_elements, BLOCK: tl.constexpr):
    # TODO (student)
    ...


@triton.jit
def swiglu_bwd_kernel(a_ptr, b_ptr, dy_ptr, da_ptr, db_ptr, n_elements, BLOCK: tl.constexpr):
    # TODO (student)
    ...


class SwiGLU(torch.autograd.Function):
    """Differentiable wrapper around swiglu_{fwd,bwd}_kernel."""

    @staticmethod
    def forward(ctx, a, b):
        # TODO (student)
        ...

    @staticmethod
    def backward(ctx, grad_out):
        # TODO (student)
        ...


def triton_swiglu(a, b):
    return SwiGLU.apply(a, b)


def torch_swiglu(a, b):
    return F.silu(a) * b


In [ ]:
for dtype, atol, rtol in [(torch.float32, 1e-5, 1e-4), (torch.bfloat16, 1e-2, 2e-2)]:
    for shape in [(999_983,), (2, 2048, 11008)]:   # odd N, LLaMA-ish MLP dims
        a = torch.randn(*shape, device=DEV, dtype=dtype)
        b = torch.randn(*shape, device=DEV, dtype=dtype)
        report(f"swiglu fwd {shape} {str(dtype).split('.')[-1]}",
               triton_swiglu(a, b), torch_swiglu(a, b), atol, rtol)

for dtype, atol, rtol in [(torch.float32, 1e-4, 1e-3), (torch.bfloat16, 2e-2, 2e-2)]:
    for shape in [(999_983,), (128, 11008)]:
        a1 = torch.randn(*shape, device=DEV, dtype=dtype, requires_grad=True)
        b1 = torch.randn(*shape, device=DEV, dtype=dtype, requires_grad=True)
        a2 = a1.detach().clone().requires_grad_(True)
        b2 = b1.detach().clone().requires_grad_(True)
        torch_swiglu(a1, b1).pow(2).sum().backward()
        triton_swiglu(a2, b2).pow(2).sum().backward()
        report(f"swiglu bwd da {shape} {str(dtype).split('.')[-1]}", a2.grad, a1.grad, atol, rtol)
        report(f"swiglu bwd db {shape} {str(dtype).split('.')[-1]}", b2.grad, b1.grad, atol, rtol)


In [ ]:
a = torch.randn(4096, 11008, device=DEV, dtype=torch.bfloat16)
b = torch.randn(4096, 11008, device=DEV, dtype=torch.bfloat16)
nbytes_fwd = 3 * a.numel() * a.element_size()   # read a and b, write out
bench_op("swiglu fwd", str(tuple(a.shape)), lambda: triton_swiglu(a, b),
         lambda: torch_swiglu(a, b), nbytes_fwd)

ag = a.clone().requires_grad_(True)
bg = b.clone().requires_grad_(True)

def triton_swiglu_fwd_bwd():
    triton_swiglu(ag, bg).sum().backward()
    ag.grad = bg.grad = None

def torch_swiglu_fwd_bwd():
    torch_swiglu(ag, bg).sum().backward()
    ag.grad = bg.grad = None

nbytes_fb = 2 * nbytes_fwd   # forward reads + backward reads/writes
bench_op("swiglu fwd+bwd", str(tuple(a.shape)), triton_swiglu_fwd_bwd, torch_swiglu_fwd_bwd, nbytes_fb)
print(f"peak memory:  eager {peak_mem_mib(torch_swiglu_fwd_bwd):8.0f} MiB   fused {peak_mem_mib(triton_swiglu_fwd_bwd):8.0f} MiB")
del a, b, ag, bg
torch.cuda.empty_cache()


## Kernel (d): cross-entropy (forward and backward)

**TODO (student):** derive the forward and backward math in this cell before implementing. You want a numerically stable fused log-softmax + NLL per row, as formulated in the handout. Backward should not materialize the full $(B, V)$ probability matrix in HBM.


In [ ]:
@triton.jit
def ce_fwd_kernel(logits_ptr, labels_ptr, loss_ptr, lse_ptr,
                  n_cols, stride_row, BLOCK: tl.constexpr):
    # TODO (student)
    ...


@triton.jit
def ce_bwd_kernel(logits_ptr, labels_ptr, lse_ptr, dlogits_ptr,
                  n_cols, stride_row, scale,
                  BLOCK: tl.constexpr):
    # scale = dloss / n_rows, so the kernel can write
    # dz = scale * (softmax(z) - onehot) in one pass.
    # TODO (student)
    ...


class FusedCrossEntropy(torch.autograd.Function):
    """Mean cross-entropy over rows. Per-row Triton kernels do the math."""

    @staticmethod
    def forward(ctx, logits, labels):
        # TODO (student)
        ...

    @staticmethod
    def backward(ctx, dloss):
        # TODO (student)
        ...


def triton_cross_entropy(logits, labels):
    return FusedCrossEntropy.apply(logits, labels)


**Correctness.** Forward is compared to `F.cross_entropy` and the gradient to autograd through the eager reference. The bf16 tests use the standard fp32-upcast path as the reference, since that is what most training code does anyway.


In [ ]:
z = torch.randn(4, 1000, device=DEV) * 3        # small, odd vocab
y = torch.randint(0, 1000, (4,), device=DEV)
report("ce fwd fp32 (odd V)", triton_cross_entropy(z, y), F.cross_entropy(z, y), 1e-5, 1e-4)

z1 = z.clone().requires_grad_(True)
z2 = z.clone().requires_grad_(True)
F.cross_entropy(z1, y).backward()
triton_cross_entropy(z2, y).backward()
report("ce bwd fp32", z2.grad, z1.grad, 1e-5, 1e-4)

zb = torch.randn(32, 32000, device=DEV, dtype=torch.bfloat16) * 2
yb = torch.randint(0, 32000, (32,), device=DEV)
report("ce fwd bf16", triton_cross_entropy(zb, yb), F.cross_entropy(zb.float(), yb), 1e-2, 1e-2)

z1 = zb.clone().requires_grad_(True)
z2 = zb.clone().requires_grad_(True)
F.cross_entropy(z1.float(), yb).backward()
triton_cross_entropy(z2, yb).backward()
report("ce bwd bf16", z2.grad, z1.grad, 2e-3, 2e-2)


In [ ]:
# realistic LLM-scale vocab (Llama-3-sized), large enough that you have to chunk over V
V = 128256
z = torch.randn(512, V, device=DEV, dtype=torch.bfloat16)
y = torch.randint(0, V, (512,), device=DEV)
report("ce fwd bf16 V=128k", triton_cross_entropy(z, y), F.cross_entropy(z.float(), y), 1e-2, 1e-2)
del z, y
torch.cuda.empty_cache()


**Benchmark.** Forward+backward end-to-end, against the eager fp32-upcast path. Also compare peak memory. In training, the eager materializations are often what forces smaller micro-batches.


In [ ]:
B, V = 4096, 128256
base = torch.randn(B, V, device=DEV, dtype=torch.bfloat16)
y = torch.randint(0, V, (B,), device=DEV)

def eager_fwd_bwd():
    z = base.detach().requires_grad_(True)
    F.cross_entropy(z.float(), y).backward()

def triton_fwd_bwd():
    z = base.detach().requires_grad_(True)
    triton_cross_entropy(z, y).backward()

nbytes = 3 * base.numel() * base.element_size()   # lower bound: read z (fwd) + read z + write dz (bwd)
bench_op("cross-entropy fwd+bwd", f"B={B}, V={V}", triton_fwd_bwd, eager_fwd_bwd, nbytes)

print(f"peak memory:  eager {peak_mem_mib(eager_fwd_bwd):8.0f} MiB   fused {peak_mem_mib(triton_fwd_bwd):8.0f} MiB")
del base, y
torch.cuda.empty_cache()


## Tuning

Once everything passes, tune your block sizes. Use `@triton.autotune` over a few `BLOCK` / `num_warps` configs (template below) or do a small manual sweep. Re-run your benchmarks and record what changed and why in the write-up at the bottom.

*Hints:* elementwise kernels (forward and backward alike) are usually insensitive beyond `BLOCK` ≈ 1024. The row-reduction kernels (RMSNorm, CE, both directions) care about `num_warps` because of intra-block reductions. Very large `BLOCK` per program hurts occupancy.


In [ ]:
# Example: retune add_kernel once it passes correctness.
#
# @triton.autotune(
#     configs=[triton.Config({"BLOCK": b}, num_warps=w)
#              for b in (512, 1024, 2048, 4096) for w in (4, 8)],
#     key=["n_elements"])
# @triton.jit
# def add_kernel(a_ptr, b_ptr, out_ptr, n_elements, BLOCK: tl.constexpr):
#     ...
#
# then re-run the add benchmark cell and note the winning config.
#
# For the row-wise kernels (RMSNorm fwd/bwd, CE fwd/bwd), key the autotune
# on "n_cols" instead and consider num_warps in (2, 4, 8, 16).
#
# NOTE: with @triton.autotune, the autotuner owns BLOCK. In Add.forward you
# must then (1) drop the explicit `BLOCK=1024` from the kernel launch, and
# (2) let the grid depend on the config the autotuner picks:
#     grid = lambda META: (triton.cdiv(n, META["BLOCK"]),)


## Results & what to submit

Run the cell below after all kernels are implemented. It prints your filled-in results table and asserts every correctness check passed. Submit this notebook with outputs on Gradescope.

**Checklist:**

- [ ] All kernels implemented, forward for (a) and forward plus backward for (b), (c), (d). Every `report(...)` prints `PASS` in fp32 and bf16, including the odd shapes and non-contiguous inputs.
- [ ] Cross-entropy works at $V = 128{,}256$. RMSNorm and SwiGLU gradients match autograd through the eager reference.
- [ ] Results table with ms, achieved GB/s, and speedup vs eager for every op you benchmarked.
- [ ] Tuning notes, either the configs you swept or what `autotune` picked, and the effect.

Answer these in a markdown cell (or your report) using your table:

1. Is each op memory-bound or compute-bound? Justify from first principles (FLOPs vs bytes moved) and from your measured GB/s vs the H100 peak.
2. Which kernel gave the largest speedup over eager, and why?
3. For cross-entropy (and RMSNorm/SwiGLU if you benchmarked fwd+bwd), how much peak memory did fusion save, and why does that matter for training at large batch/vocab sizes?
4. What did tuning change (or not), and why?
5. Where did fp32 accumulation matter most for the bf16 error behavior?


In [ ]:
results_table()

EXPECTED_CHECKS = 42   # 11 add + 14 rmsnorm + 12 swiglu + 5 cross-entropy
assert len(ALL_OK) == EXPECTED_CHECKS, (
    f"Only {len(ALL_OK)}/{EXPECTED_CHECKS} correctness checks have run. "
    "Restart & Run All so every test cell executes before submitting.")
assert all(ALL_OK.values()), "Some correctness checks are still failing. See the FAIL lines above."
print(f"\nAll {len(ALL_OK)} checks passed. Notebook ready to submit.")
